# 🛡️ Bug Bounty Copilot - Autonomous Training Pipeline (Kaggle T4 GPU)

This notebook trains an advanced offensive security reasoning model using **Unsloth (QLoRA 4-bit)** on **Qwen2.5-7B-Instruct**.
- **Dataset**: Curated PentesterLand Bug Bounty Chain-of-Thought (CoT) pairs.
- **Hardware**: Kaggle Free T4 GPU (Background Execution Supported).
- **Output**: LoRA adapter saved to `/kaggle/working/bugbounty_copilot_lora`.

In [ ]:
# 1. Install Unsloth & Core Dependencies (Optimized for T4 GPU)
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers triton xformers


In [ ]:
# 2. Check for Previous Accumulated LoRA Weights (Mount & Resume)
from unsloth import FastLanguageModel
import torch, os, glob, urllib.request, json

max_seq_length = 4096
dtype = None # Auto detection (Float16 for T4)
load_in_4bit = True

# Injected by orchestrator before push (signed direct download URLs)
LATEST_LORA_URLS = {"bugbounty_copilot_lora/README.md": "https://www.kaggleusercontent.com/kf/357515491/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..Yef-ZWUd0ZE3W3fxS2_I-A.19Av5YI7HROlPT0Vjy6RDCKTxhraW4dwUisxiO1cYnz4uRAG7di9Pe5Cn6pKAdzqB-1-6k23F2Jk_djQ4E4NJVXECW16Ux5bK5I3NPoIFYJkPVjlskpjrdRQrm-TQ86yfmpAbUkrF-8QTkdz1XjvYjPJ04jdlJxqhXHGxibe45HfJKlyXYmp3yO6A_mn5bn0tuD8YKo6zBdRbroz18oS6g6P_Yy0X2LzOrJetrdOJKtCfQmz4Uyb5J5yNRCYg4dHIyTVm-pR11eeNCk1w3TWYKlrVuDtPiZOABUem4dt9lGxJhRYVBPDgkRsPBaD7Yed5tpagIxZuZqmDpLXo_jqZliL671us9zWhPVaub73MspzLHf4y0l2QGiC1SNcpcKo5uPOuD9Vnn0GUKCH2mkifTBA2MM79s34w2GrcrynjZ2rGJNoJEcwOlQgxQGC0F0YcBQS90FZin0ve2JzUD8sXhddJ038Y3w9KPkKJNnf2M5E7kBzwkFRBxzNQTqBSz1EdcwY-YiArAg0YcEhVmTTz5hikUaMZGhhAH7Z2yuTJjqxykrQwOCWGwTslDC4ZvklzivCUrV-MbgYIk11dURUOVllqTJQd_krsrruXAVlrp2zmr_ai0Guo-1T7he5csAW.6R30Qv0nEWxl1r5eG7WLrA/bugbounty_copilot_lora/README.md", "bugbounty_copilot_lora/adapter_config.json": "https://www.kaggleusercontent.com/kf/357515491/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..uf18FKc34m5jIMIhptoVFg.1IZSsxOgw50JB9AscAio7uQWc4zptOHxLd7_ETsivcke-7IHFvBWBpETJs8exQKEVURUbj3qpreIYvPEPoSC-M5GqKRpXf_WpfB3xe3RGdinHcvXaJT36jaZyAHGCc22ztCJrGwS3U10PI3FJc3rry2cx54ONutI0moxuv4tXvn7xkegMsNXetWjl5vcO9_xKMDHmeintNzhogmxz_jh61TXsPLXCNBnv6oeeXvRgjdVKBORSpeOImQ_Q_j-loNywh_x1uo2FpuA5IVU08JZLbYXoH9qM_Efxl5KhdFnULKUZsvVLuWFRgTFs2ghTrf3TLxyUOzQvydb_CV21g1gX-q-SZ0FznN0Y0TzidgVLqJZ9zT0MlwQZRXUggcc17ySih1eHPOCh9DRMR94_2yQTkjLzo-naa5PJCIp2EKZt49FwKK5-kDfpS6ZtB6JZgzUELrslEW2ruJyy7EkYFl8R3vVDVjMDr3a_4MoEeY4CLQzxKEoB-fx3oEJDOVfV89HKg3zuo73sTRMczsocJPrXzx3EHIK0ksu2siZES_84ygP7ah7KbPBxdnMG5FrzlXphbhpWsiN0w-GY9kBmL64PTmtRiDl8dXBgeSfMpxYRKbtFYMbPhjV0eEX65hZkVsU.FLRhfHcLHq1KushvPWQSJQ/bugbounty_copilot_lora/adapter_config.json", "bugbounty_copilot_lora/adapter_model.safetensors": "https://www.kaggleusercontent.com/kf/357515491/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..39bq2v0nGyp6XbuDwYmQ_g.XNHGtVta5ApQ2_kad1V1X--vpNtjxOB4LqSgPGUBB7BuHuWVBQIn_6GXkalss-jXV8TslCtrggIBAkaVesrbmsjc2uZn-MFYarnxm-G58Yr-s_uPk-7G9DPX2d-OOvVM7PjqX1n2D5bCP8XID24xYv0GoGPUtOCmrdSpOIqOb83zzy_8NYb4KHJLYPUSUzH4Wvvt34sj5deqVUdxO-Rq3uF1SPWO6iAPQ7hmmwPs_J8ljgcFM0TP8N7R3hBUSOXgn6LoVpLSNHmludhrngX178veWiBWYyOiCjzsDMTl9M_ctUbiklpMSlXec0FGC4m6mkD5wvLTU7LOGkkteIK8EBGXvQ4R0-LaDJHyh_kRVVeURABQqIE_iijPffuBiQqgKZ6gvpDBckgXyr81x0X9ZhWkXydhtWFfqwCKTZBSPOiJzpRVXuYTDYWPBnVlZf5bJoDYp6gluuvKOlJntl0nouJ47FOzDPDHm3v6s2PoESR3dTp0-sYXCWY93HrsMcyXnwpsY1Byd2tqAwWC99wW8350Nx5T3Ya5EQm-ikKpMUKbJkBcLtV44Qk7KNQPwgLFtqXibvbs2etGQk8JZdTi83H_Z-6j9rFars8tR8V8zMBaE7u1b7YfI9-iN-npjDCD.GDTKpDfupNE8zKt8ceR-jQ/bugbounty_copilot_lora/adapter_model.safetensors", "bugbounty_copilot_lora/chat_template.jinja": "https://www.kaggleusercontent.com/kf/357515491/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..1IbXnTxXMI3jbdfDWkUwuA.XTdKPw2U64TdUj9OPflo2q7nQ0h_EOXJVvY3CFsiAdrFAqhHMc5IePzhPQ3J0gMw9_lQJQ49hC-TanCn8y5d4Gvbd3Yivnw0Z91hiV-OcHHhg3p4wrAXbfEH1uatioYzMeE5okw_eBsdTcn-oRhacyYPvx_hr2IbvNiXu0B364adl0wlyHRFotFB8DPQctownaYngkQfaRNmebte7vhM2tcixnxasc0BFqCuj2GKAfotX56u4lbZtZPsa1gTixiPifIybPLzZB47Kvkk9VdMsr61NsWtxM6S09TiRpkqws7aNQyixBhz7Gq_Fd4o3KL8iqaCsCQ7lRtKREwMm8kz9m3yjmBqTRyJMPD17493IDonQcP35XKjYuU8ZfAoAIQe_lmDUuObZYA1YTruHRp_JQghkIhmC5RTZeW5gB3iUKrSrKWlqJFE1nOqSPGkkd_DYOlV6vYWcJ4JESW8f6UbIdWy99jKhxR9kUjKxDLZQvHE7k5u_VK88CCkas3zTt_vCHucmE7X7LA5-tSy7otk0YdvEt0WRNSIf66UpNam_qLDU0qqxfHaOM8LuTWX7FUdCEC3tEJPC99XiO79CPYRhnwrLKaUjtNTYt9tq47kRgX50cMiddMnWtE7XxzpUTy_.C2aPK0p2zJp7tHLHz_7ObQ/bugbounty_copilot_lora/chat_template.jinja", "bugbounty_copilot_lora/tokenizer.json": "https://www.kaggleusercontent.com/kf/357515491/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..KO9kfxB5MQHdGP-0aGKG-g.F_m7S6OsqUIO6I-MEKtLit9SFSJZO-Kwnuke4Y2yr6sQb9TuhMO5cErh1LWcCH704E-xY6gitBpHHjUZnPn7I-pBHulwahyWq0QjtfOK9xfd_JeVrA2lsffjhNUF-OdOW1CLBKQa7yyIA3kax6780-Bx8g9q8Pzx1tA2ubmiU2lXx-e928qv1SmU9boc2Gu9h1In7o5Q1BHhaS8NML9Tf2CkKVdoVnA50zsMO5Yrkagz23_CK1WPh3nO3ZepDOgOzJORvV6VVRC8DoVL5GBiVcCP_cx_uHFElOpW39XVy5qgdfb6bB0vkBOxPbNzyVdPgV_EKZGyMMTyaq4bbAc1AAtt2XXbcWam48l5hERpvW3G5y8BHwJJ8oEQjI1CCex4Iqs2SciHQgkXrWU9ZZMBf8iIPmjOBIh9FPq0X-HpTJojjdQmVLmP4gYGXR0vWzr0LsUQCk8LIcv9wfAsdrY-WRza3EEyQgEH38VIdE2PN2j6794pRxWg0prakvnvyz6PJ3NwS0b_tbVTOdEhubMxbe2WLdaGf83RBs60aONhZyMqI1oafVce0pIcP04r5SCnrMAKwNMXbisMNbhMRVzyk2pBCw5ZmbnHnTx0QMbQhFqO8PgCw0SOJEgpjUKdDBFe.KfkzJ7UrGDLGPdWmUrh6Tg/bugbounty_copilot_lora/tokenizer.json", "bugbounty_copilot_lora/tokenizer_config.json": "https://www.kaggleusercontent.com/kf/357515491/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..TXA3tQADJWm_fFqfno0M7Q.T4ow0Kp7uQL0un7FtSq0ZLhsO0nTVxBcXcJdD61odWDu4nfFgw2PFqqI_26Zpkc8mmSqGrF-L7MnGqJHznAXl61XzlNQ-bv4mmWdeprky1xSo79k2aey8BQvDU5d2mzabscLkGDl949VRupKmMLVJO2Es9JtCSbGnjA0yDPXXLOt3ccLweOWmXw4Y0IXRbxhLsP2erHFRnGfbEOXFlvIrTsAS4z6D-iIaWxNFFZBALJH5JiUoq3m_0h-94twTWHKvygZUlMLuFmnDTTIfaEzA0Zv6_erxx-gvU7fNHjC4wWLtyj-cgSpDW_rs8FO65XYl8mToafTbTqrFqRQ2pMhWowoEWzhOtoV_9zNivT--rGRqq_Ral-XhTh1Z74U-EPU8EUUOzY6Isn-gES4e1FgI6VjLGBWMyD_FdStYzCZFZZ7G-uPVEYTuhQgEgIHcWmyOszYVJ4O8yYjNoirjbtRV3cMniABxACH5GTe3a3cxRp8ZZagb38TUFyUObJ4Np3ZK0ipugJ4U_413Ft9cOwE0_4-Kj1nrYhdHncMTbGRDWg4vcug_PjebNrOfx3EstnEhYQETwnyQXbw0SrpaZAoaLHUOLEpSUBbK0hTaG972FmhtPqVGGerKH_oQPRqNAlf.4xmCTqRJ-28Pd8UImDTGbw/bugbounty_copilot_lora/tokenizer_config.json"} # Daemon injected signed URLs

# Priority 1: Check mounted kernel sources in /kaggle/input/
possible_loras = glob.glob("/kaggle/input/**/bugbounty_copilot_lora/adapter_model.safetensors", recursive=True)
if not possible_loras:
    possible_loras = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)

# Priority 2: Direct signed URL download from partner node
if not possible_loras and LATEST_LORA_URLS:
    print(f"[*] Downloading {len(LATEST_LORA_URLS)} accumulated LoRA files from previous run...")
    lora_dir = "/kaggle/working/latest_accumulated_lora"
    os.makedirs(lora_dir, exist_ok=True)
    for fname, url in LATEST_LORA_URLS.items():
        local_f = os.path.join(lora_dir, os.path.basename(fname))
        try:
            urllib.request.urlretrieve(url, local_f)
            print(f"    -> Downloaded: {os.path.basename(fname)}")
        except Exception as e:
            print(f"    -> Warning downloading {fname}: {e}")
    if os.path.exists(os.path.join(lora_dir, "adapter_model.safetensors")):
        possible_loras = [os.path.join(lora_dir, "adapter_model.safetensors")]

# Priority 3: Fallback via CLI
if not possible_loras:
    print("[*] Searching for partner node weights via Kaggle direct output pull...")
    os.system("kaggle kernels output mmossad2324/bugbounty-copilot-trainer -p /kaggle/working/prev_acc3 >/dev/null 2>&1 || true")
    os.system("kaggle kernels output mmossad2124/bugbounty-copilot-trainer -p /kaggle/working/prev_acc1 >/dev/null 2>&1 || true")
    candidates = glob.glob("/kaggle/working/prev_acc*/**/adapter_model.safetensors", recursive=True)
    if candidates:
        candidates.sort(key=lambda x: os.path.getmtime(x), reverse=True)
        possible_loras = [candidates[0]]

loaded_from_previous = False
if possible_loras:
    prev_adapter_dir = os.path.dirname(possible_loras[0])
    print(f"[✓] FOUND PREVIOUS DAY ACCUMULATED WEIGHTS at: {prev_adapter_dir}")
    print("[*] Resuming and stacking incremental training on top of previous day progress...")
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = prev_adapter_dir,
            max_seq_length = max_seq_length,
            dtype = dtype,
            load_in_4bit = load_in_4bit,
        )
        FastLanguageModel.for_training(model)
        loaded_from_previous = True
        print("[✓] Successfully loaded previous day accumulated weights! Knowledge is compounding.")
    except Exception as e:
        print(f"[!] Could not load previous weights directly ({e}). Falling back to base model.")

if not loaded_from_previous:
    model_name = "Qwen/Qwen2.5-7B-Instruct"
    print(f"[*] Initializing fresh base model: {model_name} in 4-bit...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )
    print("[✓] Attached fresh QLoRA adapters to base model!")


In [ ]:
# 3. Verify Active LoRA Training Status
FastLanguageModel.for_training(model)
print('[✓] Model & LoRA Adapters verified in active training mode!')


In [ ]:
# 4. Load & Format Training Dataset (ChatML Format with Industrial Conveyor Specialization)
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Qwen2.5 standard
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

# Dynamic Relay Specialization with Industrial Conveyor Chunks:
ACTIVE_NODE_USER = "mmossad2124" # Daemon updates this

# Auto-detect node identity from mounted input if available
if any("mmossad2124" in p for p in glob.glob("/kaggle/input/*")):
    ACTIVE_NODE_USER = "mmossad2324"
elif any("mmossad2324" in p for p in glob.glob("/kaggle/input/*")):
    ACTIVE_NODE_USER = "mmossad2124"

if ACTIVE_NODE_USER == "mmossad2324":
    dataset_file = "active_stage2_chunk.jsonl"
    print("[*] Node Specialization: Account 3 (Stage 2: Kernel, Memory, APTs Active Chunk - 2,400 samples)")
else:
    dataset_file = "active_stage1_chunk.jsonl"
    print("[*] Node Specialization: Account 1 (Stage 1: Web, Cloud, AI Active Chunk - 2,400 samples)")

dataset_url = f"https://raw.githubusercontent.com/mmossad2124-blip/BugBounty-Copilot/main/data_pipeline/processed_datasets/{dataset_file}"
print(f"[*] Loading conveyor chunk from: {dataset_url}")
try:
    dataset = load_dataset("json", data_files=dataset_url, split="train")
except Exception as e:
    print(f"[!] Active chunk notice ({e}). Falling back to full dataset file...")
    fallback_file = "stage2_kernel_apt_arsenal.jsonl" if ACTIVE_NODE_USER == "mmossad2324" else "stage1_web_cloud_arsenal.jsonl"
    dataset = load_dataset("json", data_files=f"https://raw.githubusercontent.com/mmossad2124-blip/BugBounty-Copilot/main/data_pipeline/processed_datasets/{fallback_file}", split="train")

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"[✓] Successfully loaded {len(dataset)} elite offensive reasoning samples!")


In [ ]:
# 5. Configure SFTTrainer (Supervised Fine-Tuning)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 300,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/kaggle/working/checkpoints",
        save_steps = 100,
    ),
)
print("[✓] Trainer configured and ready!")

In [ ]:
# 6. Train the Model!
print("[*] Starting training loop on GPU...")
trainer_stats = trainer.train()
print("[✓] Training completed successfully!")

In [ ]:
# 7. Test Inference (Sample Vulnerability Reasoning)
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are an Elite Offensive Security & Bug Bounty AI Copilot."},
    {"role": "user", "content": "Target scope: api.target.com/v1/user/profile. How would you methodically test for IDOR and privilege escalation?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

In [ ]:
# 8. Save LoRA Adapters to Kaggle Output
output_dir = "/kaggle/working/bugbounty_copilot_lora"
print(f"[*] Saving trained LoRA adapter to {output_dir}...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("[✓] LoRA adapters permanently saved to output! Available in Kaggle output tab.")